# Seaquest Oxygen 4-Frame Study — Phase 2 (oxygen qualification probes)
Three supervised probes on the FROZEN `raw_hf` masked four-frame state — leakage, U→A,
U→future (H=16/32/64) — then one qualification outcome. Each step is one committed,
locally-tested script (no inline logic). Does NOT retrain the critic, build the oracle,
or do anchors/paired eval. **Use:** TOKEN (Cell 1) + Drive raw_hf.zip (Cell 2), Run-all.


In [ ]:
import torch
print('CUDA:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '-')


In [ ]:
# 1. Clone repo at the reviewed commit.
TOKEN = 'PASTE_YOUR_GITHUB_TOKEN_HERE'
import os, subprocess
D = '/content/Goal-Conditioned-Confounded-MsPacman'
if not os.path.isdir(D):
    subprocess.run(['git','clone',f'https://{TOKEN}@github.com/tingrui-huang/Goal-Conditioned-Confounded-MsPacman.git',D], check=True)
%cd /content/Goal-Conditioned-Confounded-MsPacman
!git pull -q && git log --oneline -1


In [ ]:
# 2. Load raw_hf from Drive (mount + unzip + auto-locate traj_*.npz).
import glob, os, zipfile
from google.colab import drive
drive.mount('/content/drive')
ZIP = '/content/drive/MyDrive/raw_hf.zip'     # <-- EDIT if elsewhere
assert os.path.exists(ZIP), f'zip not found at {ZIP}'
EXTRACT = '/content/raw_hf_extracted'
if not glob.glob(EXTRACT + '/**/traj_0000.npz', recursive=True):
    with zipfile.ZipFile(ZIP) as z: z.extractall(EXTRACT)
hits = glob.glob(EXTRACT + '/**/traj_0000.npz', recursive=True)
assert hits, f'no traj_*.npz inside {ZIP}'
DATA_ROOT = os.path.dirname(hits[0])
n = len(glob.glob(DATA_ROOT + '/traj_*.npz'))
print('DATA_ROOT =', DATA_ROOT, '| trajectories:', n)
assert n == 40, f'expected 40 traj, found {n}'


In [ ]:
# 3. Pre-probe assertions (shape (B,84,84,12), masking, alignment, no boundary cross, no future-oxygen target, frozen split seed 2606).
!python -m seaquest_ccrl.scripts.oxy4_assertions --root "$DATA_ROOT"


In [ ]:
# 4. Phase 2.1 — four-frame oxygen leakage (masked vs visible vs trivial baseline).
!python -m seaquest_ccrl.scripts.oxy4_probe_leakage --root "$DATA_ROOT"


In [ ]:
# 5. Phase 2.2 — conditional U->A (masked state vs +oxygen; exact-18 + semantic-12).
!python -m seaquest_ccrl.scripts.oxy4_probe_action --root "$DATA_ROOT"


In [ ]:
# 6. Phase 2.3 — conditional U->future at H=16,32,64 (state+action vs +oxygen).
!python -m seaquest_ccrl.scripts.oxy4_probe_future --root "$DATA_ROOT" --horizons 16,32,64


In [ ]:
# 7. Oxygen qualification report (one outcome).
!python -m seaquest_ccrl.scripts.oxy4_qualify
import json; print(json.dumps(json.load(open('artifacts/seaquest/oxygen_4frame/oxygen_qualification.json')), indent=2))


In [ ]:
# 8. Persist + download Phase-2 outputs (predictions, bootstrap inputs, metrics, figures).
import shutil, os, glob
shutil.make_archive('seaquest_oxygen_4frame_phase2', 'zip', 'artifacts/seaquest/oxygen_4frame')
try:
    from google.colab import drive; drive.mount('/content/drive')
    dst='/content/drive/MyDrive/seaquest_oxygen_4frame_phase2'; os.makedirs(dst, exist_ok=True)
    shutil.copy('seaquest_oxygen_4frame_phase2.zip', dst); print('copied to Drive')
except Exception as e: print('Drive optional:', e)
try:
    from google.colab import files; files.download('seaquest_oxygen_4frame_phase2.zip')
except Exception: pass


## STOP — Phase 2 only
Send `oxygen_qualification.json` for review. Do **not** start Phase 3 (oracle critic) /
anchors / paired evaluation until the qualification outcome is approved.
